In [ ]:
from pathlib import Path
import os
import numpy as np
import arviz as az
import pickle
import pandas as pd
from matplotlib_inline.backend_inline import set_matplotlib_formats
from os import listdir as ls

from emu_renewal.inputs import OUTPUTS_PATH
from emu_renewal.outputs import load_targets
from emu_renewal.plotting import plot_multianalysis_fit, plot_post_prior_comparison

set_matplotlib_formats("svg")

In [ ]:
job_path = OUTPUTS_PATH / "43514097"
countries = ls(job_path)
n_cols = 3
for c in countries:
    analysis_paths = [d[0] for d in os.walk(job_path / c)][1:]
    analysis_names = [a.split("\\")[-1] for a in analysis_paths]
    analysis_dict = {a: Path(p) for a, p in zip(analysis_names, analysis_paths)}
    
    spaghs = {a: pd.read_hdf(p / "spaghetti.h5") for a, p in analysis_dict.items()}
    no_mob_path = analysis_dict["no_mob"]
    targs = load_targets(no_mob_path)
    plot_multianalysis_fit(c, targs, spaghs)
    
    idata = az.from_netcdf(no_mob_path / "idata_filtered.nc");
    priors = pickle.load(open(no_mob_path / "priors.pkl", "rb"))
    epi_params = [p for p in priors if p != "shared_dispersion"]
    n_rows = int(np.ceil(len(priors) / n_cols))
    plot_post_prior_comparison(idata, epi_params, priors, c, req_grid=[n_rows, n_cols], req_size=[12, 15]);